# Cambrian-P: ask questions about your own video

Runs [Cambrian-P](https://github.com/cambrian-mllm/cambrian-p) on a video stored in your Google Drive.

**Before you start:** Runtime → Change runtime type → **A100 GPU**. The 7B model needs about 15 GB for its weights alone, so the free T4 is too small. An L4 (24 GB) may work with fewer frames (set `NUM_FRAMES` to 32).

**How to use**
1. Fill in the settings cell and run it.
2. Run the rest of the cells in order (Runtime → Run all works too).
3. The first time takes about 20–30 minutes (install + 15 GB model download). Run the optional backup cell once afterwards, and later sessions restore from Drive in a few minutes.

Within one session you can change the video or questions and rerun only the settings cell and the last two cells.

**Check the load worked:** the run log should contain a `token_num:` line, `[cam-load] ✓ Loaded model.rec_head.camera_tokens`, and `SUCCESS: All weights in new_state_dict match the loaded model.` Without them, the camera tokens are not being used.

In [ ]:
# @title 1. Settings
VIDEO_PATH = "/content/drive/MyDrive/videos/my_video.mp4"  # @param {type:"string"}
MODEL = "Cambrian-P-7B"  # @param ["Cambrian-P-7B", "Cambrian-P-7B-32f", "Cambrian-P-7B-Mix-MA", "Cambrian-P-7B-Mix-3R", "Cambrian-P-7B-Mix-CamS"]
NUM_FRAMES = 0  # @param {type:"integer"}
BACKUP_PATH = "/content/drive/MyDrive/cambrianp_env.tar"  # @param {type:"string"}
BACKUP_INCLUDES_MODEL = False  # @param {type:"boolean"}
REPO_URL = "https://github.com/cambrian-mllm/cambrian-p.git"  # @param {type:"string"}

# One question per entry. The model is loaded once and answers all of them.
QUESTIONS = [
    "How many chairs are in the room?",
    "Describe the path the camera takes through the scene.",
]

# NUM_FRAMES = 0 uses the frame count each model was evaluated with (doc/evaluation.md).
DEFAULT_FRAMES = {
    "Cambrian-P-7B": 128, "Cambrian-P-7B-Mix-CamS": 128,
    "Cambrian-P-7B-Mix-MA": 64,
    "Cambrian-P-7B-32f": 32, "Cambrian-P-7B-Mix-3R": 32,
}
frames = NUM_FRAMES or DEFAULT_FRAMES[MODEL]

import json, os
os.environ.update({
    "VIDEO_PATH": VIDEO_PATH, "MODEL": MODEL, "FRAMES": str(frames),
    "BACKUP_PATH": BACKUP_PATH, "BACKUP_INCLUDES_MODEL": str(int(BACKUP_INCLUDES_MODEL)),
    "REPO_URL": REPO_URL,
    # Keep uv's Python inside /content so the Drive backup contains everything the venv needs.
    "UV_PYTHON_INSTALL_DIR": "/content/python",
    "HF_HUB_ENABLE_HF_TRANSFER": "1",
})
with open("/content/questions.json", "w") as f:
    json.dump(QUESTIONS, f)
print(f"{MODEL}, {frames} frames, {len(QUESTIONS)} question(s)")

In [ ]:
# @title 2. Check the GPU and mount Google Drive
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
if not gpu:
    raise SystemExit("No GPU found. Runtime → Change runtime type → A100 GPU, then run again.")
print("GPU:", gpu)
if int(gpu.split(",")[-1].split()[0]) < 30000:
    print("Warning: under 30 GB of GPU memory. If you run out of memory, set NUM_FRAMES to 32 in cell 1.")

from google.colab import drive
drive.mount("/content/drive")
if not os.path.exists(VIDEO_PATH):
    print(f"Warning: video not found at {VIDEO_PATH}. Check the path in cell 1 (browse it in the Files panel on the left).")

In [ ]:
%%bash
# @title 3. Set up the environment (skips itself if already done)
set -e
cd /content
if /content/venv/bin/python -c "import torch" 2>/dev/null && [ -d /content/cambrian-p ]; then
    echo "Environment already set up in this session."
elif [ -f "$BACKUP_PATH" ]; then
    echo "Restoring environment from $BACKUP_PATH ..."
    tar xf "$BACKUP_PATH" -C /content
    echo "Restored."
else
    echo "Fresh install (about 10-15 minutes) ..."
    [ -d cambrian-p ] || git clone -q "$REPO_URL" cambrian-p
    pip install -q uv
    # The repo is pinned to Python 3.11, which may differ from Colab's own Python.
    uv venv --seed --python 3.11 /content/venv
    source /content/venv/bin/activate
    cd cambrian-p
    # Same steps and order as the README's "Environment Preparation".
    pip install -q -e ./vggt
    pip install -q hydra-core tensorboard iopath wcmatch fvcore
    pip install -q --upgrade pip
    pip install -q -e ".[train]"
    pip install -q torch==2.4.1+cu121 torchvision==0.19.1+cu121 torchaudio==2.4.1+cu121 \
        --index-url https://download.pytorch.org/whl/cu121
    pip install -q accelerate==0.29.3 easydict matplotlib roma evo imageio OpenEXR
    # Used by lmms-eval's model wrapper. flash-attn is skipped: the wrapper always uses PyTorch's
    # built-in attention (sdpa) on torch >= 2.1.2, so it would never be used.
    pip install -q loguru sqlitedict sacrebleu evaluate tenacity pytz
    echo "Install finished."
fi
# pyarrow: newer releases need NumPy 2, but the repo pins NumPy 1.26.1. Also fixes backups made before this pin.
/content/venv/bin/pip install -q pyarrow==15.0.2


In [ ]:
%%bash
# @title 4. Download the model (skips itself if already present)
set -e
CKPT=/content/cambrian-p/ckpts/$MODEL
if [ -f "$CKPT/config.json" ]; then
    echo "$MODEL already downloaded."
else
    echo "Downloading $MODEL (about 15 GB) ..."
    /content/venv/bin/huggingface-cli download "nyu-visionx/$MODEL" --local-dir "$CKPT"
fi

In [ ]:
%%bash
# @title 5. (Optional, run once) Save the environment to Drive for faster future sessions
# Without the model the archive is several GB; with it, add about 15 GB (needs paid Drive storage).
set -e
cd /content
EXCLUDE="--exclude=cambrian-p/ckpts"
[ "$BACKUP_INCLUDES_MODEL" = "1" ] && EXCLUDE=""
echo "Writing $BACKUP_PATH ..."
tar cf "$BACKUP_PATH" $EXCLUDE python venv cambrian-p
ls -lh "$BACKUP_PATH"

In [ ]:
%%writefile /content/run_video.py
"""Answer questions about one video with Cambrian-P, reusing the lmms-eval model wrapper."""
import argparse
import json
import os

from PIL import Image
import cambrianp.utils as cambrianp_utils
from lmms_eval.models.simple.cambrian_p import CambrianP, expand2square
from cambrianp.constants import DEFAULT_IMAGE_TOKEN, IMAGE_TOKEN_INDEX
from cambrianp.conversation import conv_templates
from cambrianp.mm_utils import tokenizer_image_token

ap = argparse.ArgumentParser()
ap.add_argument("--model_path", required=True)
ap.add_argument("--video", required=True)
ap.add_argument("--questions_file", required=True)
ap.add_argument("--num_frames", type=int, default=128)
args = ap.parse_args()

with open(args.questions_file) as f:
    questions = json.load(f)

# For any checkpoint path containing "cambrian", the loader converts the config to LLaVA-OneVision
# style (cambrianp/utils.py). That conversion is meant for starting training from Cambrian-S. Released
# Cambrian-P checkpoints were saved already converted, so converting again overwrites vocab_size
# (152064 vs the weights' 151672) and deletes the camera-head settings, which silently drops the
# per-frame camera tokens. Keep the checkpoint's config as saved, and switch the camera head on
# (use_camera_tokens only takes effect with load_rec_model, llava_arch.py:556; the pose eval sets it too).
def _keep_checkpoint_config(config, model_path):
    config.load_rec_model = True
    if not os.path.isfile(str(getattr(config, "rec_model_path", None))):
        # Path to the authors' VGGT init file; the trained head weights come from the checkpoint itself.
        config.rec_model_path = "None"
    return config

cambrianp_utils.cast_cambrian_config_to_llava_ov_style = _keep_checkpoint_config

# Flags from doc/evaluation.md.
m = CambrianP(pretrained=args.model_path, model_name="llava_qwen", conv_template="qwen_1_5",
              max_frames_num=args.num_frames, use_camera_tokens=True, camera_tokens_mode="camera_tokens",
              camera_tokens_place="append_to_frame", query_mode="query_after_image")

# Same frame preprocessing as CambrianP.generate_until for video inputs.
frames = m._load_video_frames(args.video, m.max_frames_num)
image_size = list(frames.shape[1:3])
bg = tuple(int(x * 255) for x in m._image_processor.image_mean)
frames = [expand2square(Image.fromarray(f, mode="RGB"), bg) for f in frames]
pixels = m._image_processor.preprocess(frames, return_tensors="pt")["pixel_values"].half().cuda()

pad_id = m.tokenizer.pad_token_id if m.tokenizer.pad_token_id is not None else m.tokenizer.eos_token_id
for question in questions:
    conv = conv_templates["qwen_1_5"].copy()
    conv.append_message(conv.roles[0], DEFAULT_IMAGE_TOKEN + "\n" + question)
    conv.append_message(conv.roles[1], None)
    ids = tokenizer_image_token(conv.get_prompt(), m.tokenizer, IMAGE_TOKEN_INDEX, return_tensors="pt").unsqueeze(0).to(m.device)
    answer = m._run_inference_step(ids, ids.ne(pad_id), [pixels], image_size, "video", {}, args.video)[0]
    print(f"\nQ: {question}\nA: {answer}", flush=True)

In [ ]:
%%bash
# @title 6. Run Cambrian-P on your video
source /content/venv/bin/activate
cd /content/cambrian-p
export PYTHONPATH=$PWD/lmms-eval:$PWD
python /content/run_video.py \
    --model_path "ckpts/$MODEL" \
    --video "$VIDEO_PATH" \
    --questions_file /content/questions.json \
    --num_frames "$FRAMES"